# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the FAIR^2 dataset using the `mlcroissant` library, leveraging the Croissant metadata schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Extract JSON metadata for readable summary
metadata_json = dataset.metadata.to_json()
print(f"Dataset Name: {metadata_json.get('name')}")
print(f"Description: {metadata_json.get('description')}")
print(f"Published: {metadata_json.get('datePublished')}")
print(f"License: {metadata_json.get('license')}")


## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

In [ ]:
# Helper: List all RecordSets, Fields, and Columns by @id
def entity_summary(ds):
    print("\nRecord Sets:")
    for rs in ds.metadata.record_sets:
        print(f"  Record Set Name: {getattr(rs, 'name', None)} | @id: {rs.id}")
        # List fields
        if hasattr(rs, 'fields') and rs.fields:
            print("    Fields:")
            for f in rs.fields:
                print(f"      Field: {getattr(f, 'name', None)} | @id: {getattr(f, 'id', None)}  | dataType: {getattr(f, 'data_type', None)}")
                # List columns
                if hasattr(f, 'columns') and f.columns:
                    for c in f.columns:
                        print(f"          Column: {getattr(c, 'name', None)} | @id: {getattr(c, 'id', None)}")

entity_summary(dataset)

Below we print a sample of records for a given record set using its `@id`.

In [ ]:
# Review a sample of records from the main RecordSet.
# Replace this with the primary record set @id from the overview above.
main_record_set_id = None
# Find the RecordSet that contains the main protein data
for rs in dataset.metadata.record_sets:
    if 'protein' in (getattr(rs, 'name', '').lower() or '') or 'main' in (getattr(rs, 'name', '').lower() or ''):
        main_record_set_id = rs.id
        break
if main_record_set_id is None and dataset.metadata.record_sets:
    main_record_set_id = dataset.metadata.record_sets[0].id  # fallback: first one

print(f"Using RecordSet @id: {main_record_set_id}")
# Print a few records (dicts with keys as field @id or names)
for idx, record in enumerate(dataset.records(record_set=main_record_set_id)):
    print(record)
    if idx>=2:
        break

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis. Each record set is referenced by its `@id`.

In [ ]:
# Extract all RecordSet @ids:
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]
print(f"Available record sets (@id): {record_sets_ids}")

dataframes = {}
# Load each record set into a pandas DataFrame, if not empty
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet {record_set_id} with shape {df.shape}")

# Analyze the main RecordSet from before
print(f"\nColumns in DataFrame for {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations may include removing outliers, transforming data, or grouping by attributes.

Below, we select a numeric field for analysis by its `@id`.

In [ ]:
# Choose a numeric field for analysis from the main record set. 
# We'll attempt to auto-pick a numeric field.
df = dataframes[main_record_set_id]
numeric_field_id = None
# Optionally print all field @id and guessed type
print("Numeric-like fields:")
for f in dataset.metadata.record_set(main_record_set_id).fields:
    # Many croissant fields use lower-case data_type
    dt = getattr(f, 'data_type', None)
    if dt and dt.lower() in ('integer', 'float', 'number'):
        print(f"  {f.id} (name: {f.name}, type: {dt})")
        if numeric_field_id is None:
            numeric_field_id = f.id

assert numeric_field_id is not None, "No numeric field found."
print(f"\nUsing numeric field @id for analysis: {numeric_field_id}")

# Make sure to cast the column to numeric in pandas
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Choose a threshold (median or a fixed value, e.g. 10)
threshold = df[numeric_field_id].median()
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (median): {len(filtered_df)} records")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to pick a categorical/grouping field (not the numeric one, and not an index field)
group_field_id = None
for f in dataset.metadata.record_set(main_record_set_id).fields:
    dt = getattr(f, 'data_type', None)
    if dt and dt.lower() in ('text', 'string', 'categorical') and f.id != numeric_field_id:
        group_field_id = f.id
        break

if group_field_id and group_field_id in filtered_df.columns:
    print(f"\nGrouping data by field @id: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(grouped_df.head())
else:
    print("No suitable group field found.")

## 5. Visualization

Visualize numeric data distributions and relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f'Distribution of {numeric_field_id}')
plt.show()

if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10,5))
    top_cats = df[group_field_id].value_counts().index[:6]
    sns.boxplot(
        data=df[df[group_field_id].isin(top_cats)],
        x=group_field_id, y=numeric_field_id
    )
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.show()

## 6. Conclusion

In this notebook, we explored the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells FAIR^2 dataset using the `mlcroissant` library. We loaded metadata and records by referencing `@id` values for each data entity, performed basic data extraction, filtering, normalization, grouping, and visualized the main numeric field. This approach ensures transparent, reproducible exploration directly from the Croissant schema.

You can extend this analysis by referencing additional fields or record sets as needed for your scientific or data science objectives.